# Fine-tuning YOLOv8n - Jacinthe d'eau JACIGREEN

Notebook Colab pour entrainer le modele puis exporter `best.onnx` vers `ai/models/jacinthe_v1.onnx`.

In [ ]:
# Installation des dépendances nécessaires pour le fine-tuning YOLOv8

!pip install -q ultralytics roboflow

from ultralytics import YOLO
from pathlib import Path
import os

In [ ]:
"""
Configuration du dataset JACIGREEN.

Le dataset est fourni par Roboflow et contient :
- images train
- images validation
- annotations YOLO

Le fichier data.yaml décrit :
- chemins des images
- classes détectées
"""

DATASET_PATH = "/content/Water-Hyacinth-Detection-1"

DATA_YAML = f"{DATASET_PATH}/data.yaml"

print(DATA_YAML)

In [ ]:
# Optionnel: import direct depuis Roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="TON_API_KEY")
# dataset = rf.workspace("xxx").project("water-hyacinth").version(1).download("yolov8")
from ultralytics import YOLO
from google.colab import userdata

API_KEY = userdata.get(
    "ROBOFLOW_API_KEY"
)

from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)

project = rf.workspace(
    "ismael-gvy1e"
).project(
    "water-hyacinth-detection-omno0-zt5yw"
)


dataset = project.version(
    1
).download(
    "yolov8"
)

In [ ]:
"""
Entraînement du modèle YOLOv8n.

YOLOv8n est utilisé car :
- léger
- rapide
- adapté à une API temps réel

Le modèle de base utilise le Transfer Learning
à partir des poids COCO.
"""

model = YOLO("yolov8n.pt")


results = model.train(
    data=DATA_YAML,

    # durée d'apprentissage
    epochs=50,

    # taille entrée réseau
    imgsz=640,

    # nombre images par batch
    batch=16,

    # arrêt anticipé si stagnation
    patience=10,


    # augmentation des données
    augment=True,
    mosaic=1.0,
    flipud=0.3,
    fliplr=0.5,

    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,


    # GPU Colab
    device=0,


    project="jacigreen",
    name="jacinthe_v1",

    save=True
)

In [ ]:
"""
Évaluation du modèle après entraînement.

Métriques principales :
- mAP50 : précision globale détection
- Precision : qualité des prédictions
- Recall : capacité à retrouver les objets
"""

metrics = model.val()


print(
    f"mAP50     : {metrics.box.map50:.2%}"
)

print(
    f"Precision : {metrics.box.mp:.2%}"
)

print(
    f"Recall    : {metrics.box.mr:.2%}"
)

In [ ]:
"""
Export du modèle vers ONNX.

Le backend FastAPI utilise ONNX Runtime
pour faire les prédictions en production.
"""


model.export(
    format="onnx",
    opset=12,
    simplify=True
)
# Copier ensuite jacigreen/jacinthe_v1/weights/best.onnx vers ai/models/jacinthe_v1.onnx

In [ ]:
from google.colab import files
import os

MODEL_ONNX_REEL = '/content/runs/detect/jacigreen/jacinthe_v1/weights/best.onnx'

if os.path.exists(MODEL_ONNX_REEL):
    print("[INFO] Fichier trouvé avec succès ! Lancement du téléchargement...")
    files.download(MODEL_ONNX_REEL)
else:
    print(f"[ERREUR] Impossible de trouver le fichier à l'adresse : {MODEL_ONNX_REEL}")
    print("[ASTUCE] Regarde dans la barre latérale gauche de Colab, explore les dossiers 'runs/detect/jacigreen/jacinthe_v1/weights/' pour voir où est caché ton fichier 'best.onnx'.")